##DBTC-1104

In [0]:
spark.sql("""
ALTER TABLE `p-cdh`.primaryactive.lg
ADD COLUMN LGFeePQ STRING AFTER LoadingDateTime;
"")

In [0]:
spark.sql("""
ALTER TABLE `p-cdh`.primaryactive.lg
ADD COLUMN DateOfCallPayment DATE AFTER LGFeePQ;
""")

In [0]:
spark.sql("""
MERGE INTO `p-cdh`.primaryactive.lg
USING `p-cdh`.landed.extfile_lg ext
ON ext.DebtorID = lg.CDI
AND ext.LGContract = lg.LGAgreementKey
AND ext.BusinessDate = lg.BusinessDate
WHEN MATCHED THEN
UPDATE SET
lg.LGFeePQ = ext.LGFeePQ
""")

In [0]:
spark.sql("""
MERGE INTO `p-cdh`.primaryactive.lg
USING `p-cdh`.landed.extfile_lg ext
ON ext.DebtorID = lg.CDI
AND ext.LGContract = lg.LGAgreementKey
AND ext.BusinessDate = lg.BusinessDate
WHEN MATCHED THEN
UPDATE SET
lg.DateOfCallPayment = CAST(
    CASE 
        WHEN ext.DateOfCallPayment = "0000-01-01" THEN NULL
        ELSE ext.DateOfCallPayment
    END 
AS DATE);
""")

## Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)

